# Computer Architecture and Compute for ML, plus a Java Primer

Notebook for **Chapter 13** of the *cs* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/cs/ch13.html

Tip: change a number, predict the output, then run the cell again.

## 13.1 CPU, GPU and TPU

In [ ]:
import time, numpy as np

# 1. Python loop versus numpy (vectorization)
n = 2_000_000
xs = list(range(n))
a = np.arange(n, dtype=np.float64)
t = time.perf_counter(); s1 = sum(x * x for x in xs); t1 = time.perf_counter() - t
t = time.perf_counter(); s2 = float((a * a).sum()); t2 = time.perf_counter() - t
print("python loop:", s1)
print("numpy      :", s2)
print(f"loop {t1*1000:.1f} ms, numpy {t2*1000:.1f} ms, speedup about {t1/t2:.0f}x")

# 2. Memory access pattern: copy a matrix in order vs transposed (strided reads)
m = np.random.rand(6000, 6000)
t = time.perf_counter(); m.copy(); c1 = time.perf_counter() - t
t = time.perf_counter(); np.ascontiguousarray(m.T); c2 = time.perf_counter() - t
print(f"in-order copy {c1*1000:.0f} ms, transposed copy {c2*1000:.0f} ms, ratio {c2/c1:.1f}x")

# 3. Memory-bound (add) vs compute-bound (matmul)
x = np.random.rand(3000, 3000).astype(np.float32)
t = time.perf_counter(); y = x + x; ta = time.perf_counter() - t
t = time.perf_counter(); z = x @ x; tm = time.perf_counter() - t
bytes_add = 3 * x.nbytes          # read x twice (conservatively) + write y
print(f"add   : {ta*1000:.1f} ms, {x.size/ta/1e9:.2f} GFLOP/s, about {bytes_add/ta/1e9:.0f} GB/s moved")
print(f"matmul: {tm*1000:.1f} ms, {2*3000**3/tm/1e9:.0f} GFLOP/s")
print("FLOPs per byte  add:", round(x.size/bytes_add, 3), " matmul:", round(2*3000**3/(3*x.nbytes), 1))

## 13.2 The Memory Hierarchy

## 13.3 Memory-bound versus Compute-bound

## 13.4 Reading GPU Specs: Will the Model Fit?

In [ ]:
def gb(x): return x/1e9
def fit(params_b, bytes_per_param, name):
    w=params_b*1e9*bytes_per_param
    print(f"{name}: weights = {params_b}B x {bytes_per_param} bytes = {gb(w):.1f} GB")
    return w
for p in (7,13,70):
    fit(p,2,f"{p}B fp16"); fit(p,0.5,f"{p}B 4-bit")
# training with Adam, mixed precision rule of thumb: ~16 bytes per param
for p in (1,7):
    print(f"train {p}B params, ~16 B/param: {p*16} GB (before activations)")
# KV cache: 2 * layers * kv_heads * head_dim * bytes * tokens
layers,kvh,hd,tok,b=32,8,128,4096,2
kv=2*layers*kvh*hd*b*tok
print(f"KV cache example (32 layers, 8 KV heads, dim 128, 4096 tokens, fp16): {kv/1e9:.2f} GB per sequence")
# arithmetic intensity roofline
peak=100e12; bw=1e12
print("ridge point FLOPs/byte for 100 TFLOPS, 1 TB/s:",peak/bw)
print("decode speed upper bound for 14 GB weights at 1 TB/s:",round(bw/14e9,1),"tokens/s")
# cost
hours=20
for name,rate in (("on-demand",1.00),("spot (assume 60% off)",0.40)):
    print(f"{name}: {hours}h x rate {rate:.2f} units/h = {hours*rate:.2f} units")

## 13.5 The Cost of Compute

## 13.6 Java for Data People: A Short Primer

## 13.7 Roofline Numbers, Precision Formats and Splitting a Model Across GPUs

In [ ]:
peak_flops, bw = 200e12, 2e12
ridge = peak_flops / bw

def report(name, flops, bytes_moved):
    ai = flops / bytes_moved
    t_c, t_m = flops / peak_flops, bytes_moved / bw
    kind = "compute-bound" if ai > ridge else "memory-bound"
    print(f"{name}: intensity {ai:.2f}, compute {t_c*1e6:.4f} us, memory {t_m*1e6:.4f} us -> {kind}, best time {max(t_c, t_m)*1e6:.4f} us")

print("ridge point:", ridge, "FLOP/byte")
n = 10**9
report("vector add", n, 3 * 2 * n)
m = 8192
report("matmul 8192", 2 * m**3, 3 * m**2 * 2)
report("matmul 64 (tiny)", 2 * 64**3, 3 * 64**2 * 2)

ridge point: 100.0 FLOP/byte
vector add: intensity 0.17, compute 5.0000 us, memory 3000.0000 us -> memory-bound, best time 3000.0000 us
matmul 8192: intensity 2730.67, compute 5497.5581 us, memory 201.3266 us -> compute-bound, best time 5497.5581 us
matmul 64 (tiny): intensity 21.33, compute 0.0026 us, memory 0.0123 us -> memory-bound, best time 0.0123 us


In [ ]:
params, bw = 7e9, 2e12
for name, bytes_per in [("16-bit", 2), ("8-bit", 1), ("4-bit", 0.5)]:
    print(name, "max tokens/s at batch 1:", round(bw / (params * bytes_per)))

16-bit max tokens/s at batch 1: 143
8-bit max tokens/s at batch 1: 286
4-bit max tokens/s at batch 1: 571


In [ ]:
import numpy as np
x = np.float32(70000.0)
print("FP16 of 70000:", np.float16(x))
print("FP16 of 1.001:", float(np.float16(1.001)))
print("FP16 max:", float(np.finfo(np.float16).max), " eps:", float(np.finfo(np.float16).eps))
# BF16 is the top 16 bits of an FP32 value; emulate by truncating the low 16 bits
def to_bf16(v):
    bits = np.float32(v).view(np.uint32) & np.uint32(0xFFFF0000)
    return float(bits.view(np.float32))
print("BF16 of 70000:", to_bf16(70000.0))
print("BF16 of 1.001:", to_bf16(1.001))

FP16 of 70000: inf
FP16 of 1.001: 1.0009765625
FP16 max: 65504.0  eps: 0.0009765625
BF16 of 70000: 69632.0
BF16 of 1.001: 1.0


In [ ]:
params = 7e9
per_param = 2 + 2 + 4 + 4 + 4
total = params * per_param / 1e9
print("bytes per parameter:", per_param)
print("model state GB:", total, " per GPU if sharded over 8:", total / 8)

bytes per parameter: 16
model state GB: 112.0  per GPU if sharded over 8: 14.0


In [ ]:
n, bytes_per = 8192, 2
one_head = n * n * bytes_per
print("score matrix per head, MB:", round(one_head / 1e6, 1))
print("32 heads, one layer, GB:", round(32 * one_head / 1e9, 2))
print("doubling n multiplies it by:", (2 * n) ** 2 // n ** 2)

score matrix per head, MB: 134.2
32 heads, one layer, GB: 4.29
doubling n multiplies it by: 4


## 13.8 Modern Java: Records, Streams and Virtual Threads

Java 21 is a long-term-support release, and these three features change how new Java code looks. You can run a single file with `java Main.java`, no separate compile step. The outputs below come from a real run with Java 21.0.5.

## Chapter recap

## Practice set

Problem 13.7.1 (easy) How many GB is a 3-billion-parameter model stored in float16?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/cs/ch13.html